# Twenty-minute approximate best responses across the 69-claim CFR+ run

This notebook evaluates every fourth playable policy snapshot from the 12-hour `r6_s4_h4_hp2tfhq_ss` neural CFR+ run.

- 72 snapshots become 18 evaluation targets.
- Each target receives one 20-measured-minute action-conditioned fitted-return responder run.
- One responder trainer learns both player roles.
- Monte Carlo response values are measured every two responder-training minutes.
- Completed targets are skipped on rerun. If a target was interrupted, a new `attempt_XX` directory is created without deleting the partial attempt.

The full sweep is approximately six hours of measured responder training plus evaluation overhead.

In [ ]:
import gc
import json
from pathlib import Path
import shutil
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'liars_poker').is_dir():
    raise RuntimeError('Could not locate the liars_poker repository root.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.env import rules_for_spec
from liars_poker.serialization import load_policy
from liars_poker.training.br_runs import run_best_responder

assert torch.cuda.is_available(), 'These responder runs are intended for CUDA.'
device = 'cuda'
torch.set_float32_matmul_precision('high')
print('repository:', REPO_ROOT)
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('free / total GPU GiB:', tuple(round(x / 2**30, 2) for x in torch.cuda.mem_get_info()))

In [ ]:
spec = GameSpec(
    ranks=6,
    suits=4,
    hand_size=4,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips', 'FullHouse', 'Quads'),
    suit_symmetry=True,
)
assert len(rules_for_spec(spec).claims) == 69

# Set this explicitly if automatic discovery selects the wrong run.
REFERENCE_RUN_DIR = None
EXPECTED_SNAPSHOTS = 72
SNAPSHOT_STRIDE = 4

BR_MINUTES = 20
BR_SEED = 7
BR_EVALUATE_EVERY_MINUTES = 2
BR_EVAL_EPISODES_PER_ROLE = 200_000
BR_EPISODES_PER_ROLE = 4096
BR_ROLLOUT_BATCH_SIZE = 1024

BR_TRAINER_KWARGS = {
    'state_hidden_sizes': (512, 512),
    'action_hidden_sizes': (128, 128),
    'embedding_dim': 256,
    'device': device,
    'replay_capacity': 1_000_000,
    'batch_size': 4096,
    'learning_rate': 1e-3,
    'train_steps': 100,
    'warmup_transitions': 20_000,
    'epsilon_start': 1.0,
    'epsilon_end': 0.05,
    'epsilon_decay_decisions': 500_000,
    'rollouts_per_action': 1,
    'fused_optimizer': True,
}

print('spec:', spec.to_short_str())
print('responder measured training hours:', EXPECTED_SNAPSHOTS / SNAPSHOT_STRIDE * BR_MINUTES / 60)

## Locate and select snapshots

Snapshots are ordered by their recorded measured CFR+ training time, not by filesystem ordering. The selected rows are one-indexed snapshots 4, 8, ..., 72.

In [ ]:
def read_json(path):
    return json.loads(path.read_text(encoding='utf-8'))

def matching_run_candidates():
    root = REPO_ROOT / 'artifacts' / 'cfr_plus_approx_reference_runs'
    candidates = []
    if not root.exists():
        return candidates
    for run_dir in root.iterdir():
        manifest_path = run_dir / 'manifest.json'
        snapshot_root = run_dir / 'evaluations' / 'policy_snapshots' / 'snapshots'
        if not run_dir.is_dir() or not manifest_path.exists() or not snapshot_root.exists():
            continue
        manifest = read_json(manifest_path)
        try:
            run_spec = json.loads(manifest['spec']) if isinstance(manifest.get('spec'), str) else manifest.get('spec')
        except (TypeError, json.JSONDecodeError):
            continue
        wanted = json.loads(spec.to_json())
        if run_spec != wanted:
            continue
        count = sum(path.is_dir() for path in snapshot_root.iterdir())
        candidates.append({
            'run directory': run_dir,
            'snapshots': count,
            'measured training min': float(manifest.get('measured_training_s', 0.0)) / 60.0,
            'status': manifest.get('status'),
        })
    return sorted(candidates, key=lambda row: (row['snapshots'], row['measured training min'], str(row['run directory'])))

candidates = matching_run_candidates()
if candidates:
    display(pd.DataFrame(candidates).style.format({'measured training min': '{:.2f}'}))

if REFERENCE_RUN_DIR is None:
    if not candidates:
        raise FileNotFoundError('No matching 69-claim CFR+ run was found. Set REFERENCE_RUN_DIR explicitly.')
    REFERENCE_RUN_DIR = Path(candidates[-1]['run directory'])
else:
    REFERENCE_RUN_DIR = Path(REFERENCE_RUN_DIR)

SNAPSHOT_ROOT = REFERENCE_RUN_DIR / 'evaluations' / 'policy_snapshots' / 'snapshots'
OUTPUT_ROOT = REFERENCE_RUN_DIR / 'approx_br_every_4th_20m'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('reference run:', REFERENCE_RUN_DIR)
print('output root:', OUTPUT_ROOT)

In [ ]:
snapshot_rows = []
for policy_dir in SNAPSHOT_ROOT.iterdir():
    if not policy_dir.is_dir() or not (policy_dir / 'metadata.json').exists():
        continue
    result_path = policy_dir / 'result.json'
    result = read_json(result_path) if result_path.exists() else {}
    snapshot_rows.append({
        'policy_dir': policy_dir,
        'measured_training_s': float(result.get('measured_training_s', 0.0)),
        'iteration': int(result.get('iteration', 0)),
        'snapshot_s': float(result.get('snapshot_s', np.nan)),
    })

snapshots = pd.DataFrame(snapshot_rows).sort_values(
    ['measured_training_s', 'iteration', 'policy_dir']
).reset_index(drop=True)
snapshots['snapshot number'] = np.arange(1, len(snapshots) + 1)
snapshots['CFR+ training min'] = snapshots['measured_training_s'] / 60.0

if len(snapshots) < EXPECTED_SNAPSHOTS:
    raise RuntimeError(f'Expected at least {EXPECTED_SNAPSHOTS} snapshots, found {len(snapshots)}.')
if len(snapshots) > EXPECTED_SNAPSHOTS:
    print(f'Found {len(snapshots)} snapshots; selecting from the first {EXPECTED_SNAPSHOTS}.')
snapshots = snapshots.iloc[:EXPECTED_SNAPSHOTS].copy()
selected = snapshots.iloc[SNAPSHOT_STRIDE - 1::SNAPSHOT_STRIDE].copy().reset_index(drop=True)
assert len(selected) == EXPECTED_SNAPSHOTS // SNAPSHOT_STRIDE == 18

# Confirm the saved policies really match the intended game before spending six hours.
_, loaded_spec = load_policy(str(selected.iloc[0]['policy_dir']))
assert loaded_spec == spec

selection_path = OUTPUT_ROOT / 'selected_snapshots.csv'
selected.assign(policy_dir=selected['policy_dir'].astype(str)).to_csv(selection_path, index=False)
display(selected[['snapshot number', 'CFR+ training min', 'iteration', 'policy_dir']].style.format({'CFR+ training min': '{:.2f}'}))

## Storage preflight

Approximate-BR replay remains in GPU memory and is not serialized. Each completed target stores a small final responder policy plus logs. No additional full CFR+ checkpoint is created.

In [ ]:
free_gib = shutil.disk_usage(REFERENCE_RUN_DIR).free / 2**30
existing_output_gib = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob('*') if path.is_file()) / 2**30
storage = pd.Series({
    'selected snapshots': len(selected),
    'measured responder-training hours': len(selected) * BR_MINUTES / 60.0,
    'existing BR output GiB': existing_output_gib,
    'currently free disk GiB': free_gib,
})
display(storage.to_frame('value').style.format(precision=3))
if free_gib < 0.5:
    raise RuntimeError('Less than 0.5 GiB free. Clean the disk before starting the sweep.')

## Run the restartable sweep

Set `RUN_SWEEP = True`. Rerunning this cell skips completed targets. An interrupted target starts a fresh numbered attempt; it does not delete or append to the partial run.

In [ ]:
def attempt_directories(target_root):
    return sorted(path for path in target_root.glob('attempt_*') if path.is_dir())

def completed_attempt(target_root):
    for attempt in reversed(attempt_directories(target_root)):
        if (attempt / 'metrics.json').exists() and (attempt / 'evaluations.jsonl').exists():
            return attempt
    return None

def next_attempt_directory(target_root):
    existing = attempt_directories(target_root)
    number = 1 + max((int(path.name.split('_')[-1]) for path in existing), default=0)
    return target_root / f'attempt_{number:02d}'

def target_output_root(row):
    return OUTPUT_ROOT / f"snapshot_{int(row['snapshot number']):03d}"

RUN_SWEEP = False

if RUN_SWEEP:
    for position, row in selected.iterrows():
        target_root = target_output_root(row)
        target_root.mkdir(parents=True, exist_ok=True)
        done = completed_attempt(target_root)
        if done is not None:
            print(
                f"[{position + 1:02d}/{len(selected)}] snapshot {int(row['snapshot number'])}: "
                f'skipping completed {done.name}'
            )
            continue

        attempt_dir = next_attempt_directory(target_root)
        kwargs = dict(BR_TRAINER_KWARGS)
        kwargs['seed'] = BR_SEED
        print(
            f"\n[{position + 1:02d}/{len(selected)}] snapshot {int(row['snapshot number'])} "
            f"at CFR+ minute {float(row['CFR+ training min']):.2f} -> {attempt_dir.name}"
        )
        result = run_best_responder(
            row['policy_dir'],
            method='action_conditioned_fitted_return',
            minutes=BR_MINUTES,
            trainer_kwargs=kwargs,
            episodes_per_role=BR_EPISODES_PER_ROLE,
            rollout_batch_size=BR_ROLLOUT_BATCH_SIZE,
            evaluate_every_minutes=BR_EVALUATE_EVERY_MINUTES,
            eval_episodes_per_role=BR_EVAL_EPISODES_PER_ROLE,
            exact_evaluation=False,
            run_dir=attempt_dir,
            debug=True,
        )
        del result
        gc.collect()
        torch.cuda.empty_cache()

    print('\nSweep complete.')

## Aggregate completed results

For each target, the reported estimate uses the best observed first-player and second-player response values over its 20-minute responder trajectory. This is a discovered lower bound on true exploitability, not an exact exploitability value.

In [ ]:
def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line]

curve_frames = []
summary_rows = []
for _, row in selected.iterrows():
    attempt = completed_attempt(target_output_root(row))
    if attempt is None:
        continue
    frame = pd.DataFrame(read_jsonl(attempt / 'evaluations.jsonl'))
    if frame.empty:
        continue
    frame = frame.sort_values('measured_training_s').copy()
    frame['best p_first'] = frame['p_first'].cummax()
    frame['best p_second'] = frame['p_second'].cummax()
    frame['best p_first LCB'] = frame['p_first_lcb'].cummax()
    frame['best p_second LCB'] = frame['p_second_lcb'].cummax()
    frame['best discovered estimate'] = frame['best p_first'] + frame['best p_second'] - 1.0
    frame['conservative lower bound'] = frame['best p_first LCB'] + frame['best p_second LCB'] - 1.0
    frame['snapshot number'] = int(row['snapshot number'])
    frame['CFR+ training min'] = float(row['CFR+ training min'])
    frame['snapshot iteration'] = int(row['iteration'])
    frame['attempt_dir'] = str(attempt)
    curve_frames.append(frame)
    final = frame.iloc[-1]
    summary_rows.append({
        'snapshot number': int(row['snapshot number']),
        'CFR+ training min': float(row['CFR+ training min']),
        'snapshot iteration': int(row['iteration']),
        'responder iterations': int(final['iteration']),
        'responder training min': float(final['measured_training_s']) / 60.0,
        'best p_first': float(final['best p_first']),
        'best p_second': float(final['best p_second']),
        'best discovered estimate': float(final['best discovered estimate']),
        'conservative lower bound': float(final['conservative lower bound']),
        'attempt directory': str(attempt),
    })

if not summary_rows:
    raise RuntimeError('No completed responder runs found yet.')

br_curves = pd.concat(curve_frames, ignore_index=True)
br_summary = pd.DataFrame(summary_rows).sort_values('snapshot number').reset_index(drop=True)
br_curves.to_csv(OUTPUT_ROOT / 'combined_evaluation_curves.csv', index=False)
br_summary.to_csv(OUTPUT_ROOT / 'snapshot_summary.csv', index=False)
display(br_summary.style.format({
    'CFR+ training min': '{:.2f}',
    'responder training min': '{:.2f}',
    'best p_first': '{:.6f}',
    'best p_second': '{:.6f}',
    'best discovered estimate': '{:.6f}',
    'conservative lower bound': '{:.6f}',
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.2))

axes[0].plot(
    br_summary['CFR+ training min'],
    br_summary['best discovered estimate'],
    marker='o',
    label='best discovered estimate',
)
axes[0].plot(
    br_summary['CFR+ training min'],
    br_summary['conservative lower bound'],
    marker='o',
    label='conservative MC lower bound',
)
axes[0].set(
    title='Approximate exploitability over CFR+ training',
    xlabel='Measured CFR+ training minutes',
    ylabel='Discovered exploitability',
)
axes[0].grid(alpha=0.3)
axes[0].legend()

colour_map = plt.cm.viridis
training_min = br_curves['CFR+ training min']
normalizer = plt.Normalize(training_min.min(), training_min.max())
for snapshot_min, frame in br_curves.groupby('CFR+ training min', sort=True):
    axes[1].plot(
        frame['measured_training_s'] / 60.0,
        frame['best discovered estimate'],
        color=colour_map(normalizer(snapshot_min)),
        alpha=0.8,
    )
scalar = plt.cm.ScalarMappable(norm=normalizer, cmap=colour_map)
fig.colorbar(scalar, ax=axes[1], label='CFR+ snapshot training minutes')
axes[1].set(
    title='Responder compute curves',
    xlabel='Measured responder-training minutes',
    ylabel='Best discovered exploitability',
)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

Interpret downward movement across CFR+ snapshots as evidence of policy improvement only when the 20-minute responder curves are reasonably close to plateauing. If later snapshots still have steeply rising responder curves, evaluator strength is limiting the comparison.